# CRISP-DM: Case-based recommendation (Medical classification dataset)

## Business framing

We already have **Phase 3** matrices (`data/processed/medical`) and **Phase 4** supervised models (`01c_Modeling_Medical`).  
This add-on answers a different question: *"Patients like this one — what happened historically?"*

**Recommendation here means:** retrieve the **nearest historical encounters** in feature space and summarize their **Result** (negative / positive).  
This is **content-based / case-based reasoning**, not collaborative filtering (there is no user–item rating matrix).

**Decision support (not autonomous care):** outputs support clinicians as **similarity context** alongside model probabilities; they do not replace protocol.


## Five-supervisor gate

| Voice | What they check | Pass criterion |
|-------|-----------------|----------------|
| **1. Domain / ethics** | Is "similar patients" interpretable on vitals + labs? | Features are clinical; narrative avoids implying causality. |
| **2. Data / leakage** | Neighbors drawn only from **train**? | Queries use **test** (or future rows) only; no test rows in the neighbor index. |
| **3. Method** | Distance metric & scaling consistent with prep? | Use **processed** CSVs from `01b` (already scaled continuous + flags). |
| **4. Evaluation** | Does neighbor consensus add signal vs chance? | Report accuracy / recall vs hold-out labels; compare to deployed classifier where useful. |
| **5. MLOps** | Reproducible artifact + metadata? | Save `Neighbor_bundle` (fitted index + config) and JSON sidecar with `k`, paths, metrics.


## 1. Imports


In [ ]:
import json
import warnings
from pathlib import Path

import joblib
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.metrics import accuracy_score, recall_score, f1_score, confusion_matrix
from sklearn.neighbors import NearestNeighbors

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted")


## 2. Load processed data and metadata

Paths resolve from the project root or from `classification_Medical_data _set/`.


In [ ]:
def _processed_dir() -> Path:
    start = Path.cwd().resolve()
    for p in [start, *list(start.parents)[:10]]:
        d = p / "data" / "processed" / "medical"
        if d.is_dir() and (d / "X_train.csv").is_file():
            return d
    raise FileNotFoundError(
        "Could not find data/processed/medical/X_train.csv. "
        "Run from project root or from classification_Medical_data _set (not only recommendation_model)."
    )


data_dir = _processed_dir()
meta_path = data_dir / "metadata.json"
meta = json.loads(meta_path.read_text(encoding="utf-8")) if meta_path.exists() else {}

X_train = pd.read_csv(data_dir / "X_train.csv")
X_test = pd.read_csv(data_dir / "X_test.csv")
y_train = pd.read_csv(data_dir / "y_train.csv").squeeze("columns")
y_test = pd.read_csv(data_dir / "y_test.csv").squeeze("columns")

if isinstance(meta.get("train_shape"), list):
    assert X_train.shape[0] == meta["train_shape"][0], "Train rows mismatch metadata."
if isinstance(meta.get("test_shape"), list):
    assert X_test.shape[0] == meta["test_shape"][0], "Test rows mismatch metadata."

print("Processed dir:", data_dir.resolve())
print("X_train", X_train.shape, "| X_test", X_test.shape)
print("Target mapping:", meta.get("target_mapping", "(see 01b)"))


## 3. Build neighbor index (train only)

`NearestNeighbors` is fit on **X_train** only. At query time we pass **X_test** rows and recover training indices + distances.

`K_NEIGHBORS` is a policy knob: larger *k* smooths noise; smaller *k* tracks local neighborhoods.


In [ ]:
K_NEIGHBORS = 15
METRIC = "euclidean"

nn = NearestNeighbors(n_neighbors=K_NEIGHBORS, metric=METRIC, algorithm="auto")
nn.fit(X_train)
distances, neighbor_idx = nn.kneighbors(X_test, n_neighbors=K_NEIGHBORS)

print("Neighbor search done:", neighbor_idx.shape)


## 4. Neighbor-majority recommendation (label = 0/1)


In [ ]:
def majority_vote_from_neighbors(idx_mat: np.ndarray) -> np.ndarray:
    labs = y_train.iloc[idx_mat.ravel()].to_numpy().reshape(idx_mat.shape)
    return (labs.mean(axis=1) >= 0.5).astype(int)


y_rec = majority_vote_from_neighbors(neighbor_idx)

acc = accuracy_score(y_test, y_rec)
rec_pos = recall_score(y_test, y_rec, pos_label=1)
f1 = f1_score(y_test, y_rec)

print(f"Neighbor majority (k={K_NEIGHBORS}) — accuracy: {acc:.4f} | recall (positive): {rec_pos:.4f} | F1: {f1:.4f}")
print("Confusion matrix [rows=true, cols=pred]:")
print(confusion_matrix(y_test, y_rec))


## 5. Optional: compare to deployed classifier probability

Side-by-side view: **similar-case consensus** vs **gradient boosting** (or whichever model `medical_best_model.pkl` encodes).


In [ ]:
model_dir = data_dir.parent.parent.parent / "models"
model_path = model_dir / "medical_best_model.pkl"

if model_path.is_file():
    clf = joblib.load(model_path)
    p_pos = clf.predict_proba(X_test)[:, 1]
    y_rule = (p_pos >= 0.5).astype(int)
    print(
        "Classifier hold-out — acc:",
        f"{accuracy_score(y_test, y_rule):.4f}",
        "| recall (pos):",
        f"{recall_score(y_test, y_rule, pos_label=1):.4f}",
    )
else:
    clf, p_pos, y_rule = None, None, None
    print("No medical_best_model.pkl found; skip classifier contrast.")


## 5b. Five-supervisor critique (this run)

| Voice | Finding on your results |
|-------|-------------------------|
| **1. Domain / ethics** | Neighbor consensus is **interpretable** (similar vitals/labs → similar outcomes). **Caveat:** 23 false negatives on hold-out (`y_rec` vs `y_test`) — in screening, FNs are the sensitive failure mode; the deployed GBM (~99% recall) is **closer** to that priority than raw k-NN vote unless you tune *k* or threshold on `neighbor_frac_positive`. |
| **2. Data / leakage** | Neighbors come **only from train**; test rows are queries only. **Pass.** |
| **3. Method** | Euclidean on **pre-scaled** `01b` features is consistent. **Gap:** votes are **unweighted** (far neighbors count as much as near ones). Rows with **large `mean_distance`** (e.g. ~46 in your table) live on the **fringes** of the training cloud — interpret cautiously. **Improvement:** distance-weighted vote or smaller *k* in sparse regions. |
| **4. Evaluation** | Neighbor majority (~87.5% acc, ~86% recall pos) is **solid** but **below** the gradient boosting hold-out. That is expected: GBM fits a smoother decision boundary. Use neighbors for **transparency** (“who were the similar cases?”), GBM for **primary risk score** if governance agrees. |
| **5. MLOps** | Bundle + sidecar already capture *k*, metric, and aggregate metrics. **Add:** plot snapshots or notebook outputs to release notes so reviewers see **confusion matrices** and **distance diagnostics** without re-running. |

**Bottom line:** Keep both artifacts — **classifier for performance**, **neighbor engine for explainability** — and flag high-`mean_distance` queries in any UI.


In [ ]:
# --- Neighbor fraction (score-like) for all test rows ---
labs_nn = y_train.iloc[neighbor_idx.ravel()].to_numpy().reshape(neighbor_idx.shape)
neighbor_frac_positive = labs_nn.mean(axis=1)
mean_dist = distances.mean(axis=1)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# (1) Confusion: neighbor majority
cm_rec = confusion_matrix(y_test, y_rec)
sns.heatmap(
    cm_rec,
    annot=True,
    fmt="d",
    cmap="Blues",
    ax=axes[0, 0],
    xticklabels=["Pred 0", "Pred 1"],
    yticklabels=["True 0", "True 1"],
)
axes[0, 0].set_title(f"Neighbor majority (k={K_NEIGHBORS})\nacc={accuracy_score(y_test, y_rec):.3f}")

# (2) Confusion: classifier (if available)
if p_pos is not None:
    cm_clf = confusion_matrix(y_test, y_rule)
    sns.heatmap(
        cm_clf,
        annot=True,
        fmt="d",
        cmap="Greens",
        ax=axes[0, 1],
        xticklabels=["Pred 0", "Pred 1"],
        yticklabels=["True 0", "True 1"],
    )
    axes[0, 1].set_title(
        "Deployed classifier (0.5 threshold)\n"
        f"acc={accuracy_score(y_test, y_rule):.3f} | recall+={recall_score(y_test, y_rule, pos_label=1):.3f}"
    )
else:
    axes[0, 1].set_visible(False)

# (3) Scalar metrics comparison
names = ["Accuracy", "Recall (+)", "F1"]
rec_vals = [
    accuracy_score(y_test, y_rec),
    recall_score(y_test, y_rec, pos_label=1),
    f1_score(y_test, y_rec),
]
x = np.arange(len(names))
w = 0.35
axes[1, 0].bar(x - w / 2, rec_vals, w, label="Neighbor vote", color="steelblue")
if p_pos is not None:
    clf_vals = [
        accuracy_score(y_test, y_rule),
        recall_score(y_test, y_rule, pos_label=1),
        f1_score(y_test, y_rule),
    ]
    axes[1, 0].bar(x + w / 2, clf_vals, w, label="Classifier", color="seagreen")
axes[1, 0].set_xticks(x)
axes[1, 0].set_xticklabels(names)
axes[1, 0].set_ylim(0, 1.05)
axes[1, 0].legend()
axes[1, 0].set_title("Hold-out metrics (same split)")

# (4) Neighborhood homogeneity vs distance (trust diagnostic)
sc = axes[1, 1].scatter(
    mean_dist,
    neighbor_frac_positive,
    c=y_test.to_numpy(),
    cmap="coolwarm",
    alpha=0.65,
    s=28,
    edgecolors="k",
    linewidths=0.2,
)
axes[1, 1].axhline(0.5, color="gray", ls="--", lw=1)
axes[1, 1].set_xlabel("Mean distance to k neighbors")
axes[1, 1].set_ylabel("Fraction of neighbors with positive label")
axes[1, 1].set_title("Local consensus vs geometry (color = true y)")
cbar = plt.colorbar(sc, ax=axes[1, 1])
cbar.set_label("true_y")

plt.tight_layout()
plt.show()


## 7. Inspection: example queries (interpretability)


In [ ]:
EXAMPLES = min(8, len(X_test))
rows = []
for i in range(EXAMPLES):
    idx = neighbor_idx[i]
    labs = y_train.iloc[idx].to_numpy()
    frac_pos = labs.mean()
    row = {
        "test_row": i,
        "true_y": int(y_test.iloc[i]),
        "neighbor_frac_positive": round(float(frac_pos), 3),
        "y_rec": int(y_rec[i]),
        "mean_distance": round(float(distances[i].mean()), 4),
    }
    if p_pos is not None:
        row["model_p_positive"] = round(float(p_pos[i]), 3)
        row["y_model"] = int(y_rule[i])
    rows.append(row)

display(pd.DataFrame(rows))


## 8. Export bundle for reuse


In [ ]:
bundle_dir = data_dir.parent.parent.parent / "recommendation_model"
bundle_dir.mkdir(parents=True, exist_ok=True)

bundle = {
    "nearest_neighbors": nn,
    "feature_columns": list(X_train.columns),
    "k_neighbors": K_NEIGHBORS,
    "metric": METRIC,
    "y_train_reference": y_train.reset_index(drop=True),
}

joblib.dump(bundle, bundle_dir / "medical_case_based_neighbor_bundle.pkl")

sidecar = {
    "artifact": "medical_case_based_neighbor_bundle.pkl",
    "k_neighbors": K_NEIGHBORS,
    "metric": METRIC,
    "processed_data_dir": str(data_dir.resolve()),
    "holdout_neighbor_majority": {
        "accuracy": float(acc),
        "recall_positive": float(rec_pos),
        "f1": float(f1),
    },
    "target_mapping": meta.get("target_mapping"),
}
(bundle_dir / "medical_recommendation_sidecar.json").write_text(
    json.dumps(sidecar, indent=2),
    encoding="utf-8",
)

print("Exported to:", bundle_dir.resolve())
